# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR⁲ dataset using the `mlcroissant` library and referencing data by their Croissant `@id` fields for transparency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata and inspect key high-level details
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")
print(f"Published on: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, the fields contained, and their `@id`s.

**Note:** All references to resources are by their `@id` per Croissant best practices.

In [ ]:
# Inspect record sets and their fields, displaying each @id

from pprint import pprint
record_sets = list(metadata.recordSets) if hasattr(metadata, "recordSets") else []
if not record_sets:
    # For mlcroissant<0.7.0 compatibility
    record_sets = list(metadata.recordSet)
print(f"Available record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}\n    @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        id_ = field.id if hasattr(field, 'id') else field['@id']
        print(f"        - {field.name} (id: {id_})")
    print("")

### Example record from a chosen record set
Print a sample record by `@id` for illustration (you can change the `record_set_id` for another set).

In [ ]:
# Choose a record set @id for example (here we will use the first one)
record_set_id = record_sets[0].id

print(f"First few records from record set {record_set_id}:")
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    pprint(record)
    if i >= 2:
        break

## 3. Data Extraction
Load tabular data from selected record sets into pandas DataFrames for analysis.

- Use the `@id` values from above for `record_sets`.
- All field and column operations should reference their `@id`.


In [ ]:
# Extract data from all record sets into dataframes by their @id

# List all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for {rs_id}\n")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping on a selected DataFrame.


In [ ]:
# Select a record set and numeric field by their @id
# You may need to inspect output from previous cells to pick the most appropriate ones.

# Example: Assume the main record set's id and fields
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Find numeric fields (by inspecting the DataFrame's columns and types)
numeric_candidates = df.select_dtypes(include=["int", "float"]).columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    # Try to coerce known-numeric columns (commonly named fields)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_candidates = df.select_dtypes(include=["int", "float"]).columns.tolist()
    numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]
threshold = df[numeric_field_id].dropna().mean() # example threshold: mean
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head(2))

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(2))

# Optional grouping by a categorical field if available
group_candidates = df.select_dtypes(include=["object", "category"]).columns.tolist()
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head(2))

## 5. Visualization
Visualize the distribution of a numeric field or its relationship to a categorical field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id} (> mean)")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field exists, display a boxplot
if "group_field_id" in locals():
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you have loaded and explored the FAIR⁲ Croissant dataset, referencing all entities via their `@id`, and examined and transformed key fields. You can further extend this analysis by referencing documentation in the Croissant schema for deeper field meanings and by leveraging more advanced modeling techniques.